In [ ]:
import math
from typing import Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
from HyenaBase.configuration_hyena import HyenaConfig
from HyenaBase.modeling_hyena import HyenaDNAPreTrainedModel, HyenaEmbeddings
from transformers.modeling_outputs import (BaseModelOutputWithNoAttention,
                                           CausalLMOutput)

# 新增RMSNorm实现
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class PGC(nn.Module):
    def __init__(self, d_model, expansion_factor=1.0, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.expansion_factor = expansion_factor
        self.dropout = dropout
        expaned_dim=int(d_model * expansion_factor)
        self.conv = nn.Conv1d(
            expaned_dim, expaned_dim, kernel_size=3, 
            padding=1, groups=expaned_dim
        )
        self.in_proj = nn.Linear(d_model, expaned_dim*2)
        self.norm = RMSNorm(expaned_dim)
        self.in_norm = RMSNorm(d_model)
        self.out_proj = nn.Linear(expaned_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, u):
        xv = self.in_proj(self.in_norm(u))
        x, v = xv.chunk(2, dim=-1)
        x_conv = self.conv(x.transpose(-1, -2)).transpose(-1, -2)
        gate = v * x_conv
        x = self.norm(gate)
        return self.dropout(self.out_proj(x))

class DropoutNd(nn.Module):
    def __init__(self, p: float = 0.5, tie=True, transposed=True):
        super().__init__()
        self.p = p
        self.tie = tie
        self.transposed = transposed

    def forward(self, X):
        if self.training and self.p > 0:
            shape = X.shape
            if not self.transposed: 
                X = rearrange(X, 'b ... d -> b d ...')
            
            mask_shape = (shape[0], shape[1]) + (1,)*(X.ndim-2) if self.tie else X.shape
            mask = torch.rand(*mask_shape, device=X.device) < (1 - self.p)
            
            X = X * mask * (1.0 / (1 - self.p))
            
            if not self.transposed: 
                X = rearrange(X, 'b d ... -> b ... d')
        return X

class S4DKernel(nn.Module):
    def __init__(self, d_model, N=64, dt_min=0.001, dt_max=0.1, lr=None):
        super().__init__()
        H = d_model
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        
        C = torch.randn(H, N//2, dtype=torch.cfloat)
        self.C = nn.Parameter(torch.view_as_real(C))
        
        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", torch.log(0.5 * torch.ones(H, N//2)), lr)
        self.register("A_imag", math.pi * repeat(torch.arange(N//2), 'n -> h n', h=H), lr)

    def forward(self, L):
        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag
        
        dtA = A * dt.unsqueeze(-1)
        K = dtA.unsqueeze(-1) * torch.arange(L, device=A.device)
        C = C * (torch.exp(dtA) - 1.) / A
        
        K = 2 * torch.einsum('hn, hnl -> hl', C, torch.exp(K)).real
        return K

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None: optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

class S4D(nn.Module):
    def __init__(self, d_model, d_state=64, dropout=0.0, transposed=True, **kernel_args):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.d_output = self.h
        self.transposed = transposed
        
        self.D = nn.Parameter(torch.randn(self.h))
        self.kernel = S4DKernel(self.h, N=self.n, **kernel_args)
        self.activation = nn.GELU()
        self.dropout = DropoutNd(dropout) if dropout > 0 else nn.Identity()
        
        self.output_linear = nn.Sequential(
            nn.Conv1d(self.h, 2*self.h, kernel_size=1),
            nn.GLU(dim=-2),
        )

    def forward(self, u, **kwargs):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)
        
        # Compute SSM Kernel
        k = self.kernel(L=L)
        
        # FFT Convolution
        k_f = torch.fft.rfft(k, n=2*L)
        u_f = torch.fft.rfft(u, n=2*L)
        y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L]
        
        # Add skip connection
        y = y + u * self.D.unsqueeze(-1)
        
        y = self.dropout(self.activation(y))
        y = self.output_linear(y)
        
        if not self.transposed:
            y = y.transpose(-1, -2)
        return y

class Lyra(nn.Module):
    def __init__(
        self,
        model_dimension,
        pgc_configs,
        num_s4,
        d_input,
        d_output=10,
        dropout=0.2,
        prenorm=True,
        final_dropout=0.2
    ):
        super().__init__()
        self.encoder = nn.Linear(d_input, model_dimension)
        
        # 修正PGC层初始化
        self.pgc_layers = nn.ModuleList()
        for pgc_hidden, num_layers in pgc_configs:
            expansion_factor = pgc_hidden / (2 * model_dimension)
            for _ in range(num_layers):
                self.pgc_layers.append(
                    PGC(model_dimension, expansion_factor, dropout)
                )
        
        self.prenorm = prenorm
        
        # S4层堆叠
        self.s4_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(num_s4):
            self.s4_layers.append(
                S4D(model_dimension, dropout=dropout, transposed=True, lr=0.001)
            )
            self.norms.append(RMSNorm(model_dimension))
            self.dropouts.append(DropoutNd(dropout, transposed=True))
        
        self.decoder = nn.Linear(model_dimension, d_output)
        # self.final_dropout = nn.Dropout(final_dropout)

    def forward(self, x, return_embeddings=False):

        x = self.encoder(x)  # (B, L, d_input) -> (B, L, d_model)

        for pgc_layer in self.pgc_layers:
            x = pgc_layer(x)
            
        x = x.transpose(-1, -2)  # (B, L, d_model) -> (B, d_model, L)
        
        for layer, norm, dropout in zip(self.s4_layers, self.norms, self.dropouts):
            z = x
            if self.prenorm:
                z = norm(z.transpose(-1, -2)).transpose(-1, -2)
            z = layer(z)
            z = dropout(z)
            x = z + x
            if not self.prenorm:
                x = norm(x.transpose(-1, -2)).transpose(-1, -2)
        
        x = x.transpose(-1, -2)  # (B, d_model, L) -> (B, L, d_model)
        embeddings = x
        return embeddings,[]
        # x = x.mean(dim=1)  # (B, d_model)
        # x = self.final_dropout(x)
        # x = self.decoder(x)
        
        # return (x, embeddings) if return_embeddings else x

class LyraDNAModel(HyenaDNAPreTrainedModel):
    def __init__(self, config, **kwargs) -> None:
        super().__init__(config, **kwargs)
        self.embeddings = HyenaEmbeddings(config)
        self.backbone = Lyra(
                model_dimension=config.d_model,
                pgc_configs=[(config.d_model, config.n_layer)],  # (hidden_dim, num_layers)
                num_s4=config.depths,
                d_input=config.d_model,
                d_output=config.vocab_size,
                dropout=config.embed_dropout
            )
        self.config = config

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
        self, input_ids, inputs_embeds=None, output_hidden_states=None, return_dict=None
    ):
        output_hidden_states = (
            output_hidden_states
            if output_hidden_states is not None
            else self.config.output_hidden_states
        )
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )
        hidden_states = self.embeddings(input_ids)
        hidden_states, all_hidden_states = self.backbone(
            hidden_states,
        )
        if return_dict:
            return BaseModelOutputWithNoAttention(
                last_hidden_state=hidden_states,
                hidden_states=all_hidden_states if output_hidden_states else None,
            )
        elif output_hidden_states:
            return hidden_states, all_hidden_states
        else:
            return hidden_states

class LyraDNAForCausalLM(HyenaDNAPreTrainedModel):

    def __init__(self, config, **kwargs):
        super().__init__(config, **kwargs)
        self.lyra = LyraDNAModel(config)
        vocab_size = config.vocab_size
        if vocab_size % config.pad_vocab_size_multiple != 0:
            vocab_size += config.pad_vocab_size_multiple - (
                vocab_size % config.pad_vocab_size_multiple
            )
        self.vocab_size = vocab_size
        self.lm_head = nn.Linear(config.d_model, vocab_size, bias=False)

        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self):
        return self.lyra.embeddings.word_embeddings

    def set_input_embeddings(self, value):
        self.lyra.embeddings.word_embeddings = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_decoder(self, decoder):
        self.lyra = decoder

    def get_decoder(self):
        return self.lyra

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, CausalLMOutput]:

        output_hidden_states = (
            output_hidden_states
            if output_hidden_states is not None
            else self.config.output_hidden_states
        )
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        # decoder outputs consists of (dec_features, layer_state, dec_hidden, dec_attn)
        outputs = self.lyra(
            input_ids=input_ids,
            inputs_embeds=inputs_embeds,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        hidden_states = outputs[0]
        logits = self.lm_head(hidden_states)
        logits = logits.float()

        loss = None
        if labels is not None:
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            # Flatten the tokens
            loss_fct = nn.CrossEntropyLoss()
            shift_logits = shift_logits.view(-1, self.vocab_size)
            shift_labels = shift_labels.view(-1)
            # Enable model parallelism
            shift_labels = shift_labels.to(shift_logits.device)
            loss = loss_fct(shift_logits, shift_labels)

        if not return_dict:
            output = (logits,) + outputs[1:]
            return (loss,) + output if loss is not None else output

        return CausalLMOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
        )

/home/zhengyulong/miniconda3/lib/python3.9/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# 超参数
batch_size = 32
seq_len = 128
input_dim = 64
lr = 1e-3
epochs = 10

# 创建示例数据
x = torch.randn(1000, seq_len, input_dim)
y = torch.randint(0, 10, (1000,))
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 初始化模型
model = Lyra(
    model_dimension=128,
    pgc_configs=[(256, 2)],  # (hidden_dim, num_layers)
    num_s4=4,
    d_input=input_dim,
    d_output=10,
    dropout=0.2
).to("cuda:1")

# 训练配置
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# 训练循环
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for inputs, targets in loader:
        optimizer.zero_grad()
        outputs = model(inputs.to("cuda:1"))
        loss = criterion(outputs.to("cuda:1"), targets.to("cuda:1"))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")


Epoch 1/10 | Loss: 2.3439
Epoch 2/10 | Loss: 2.3149
Epoch 3/10 | Loss: 2.2581
Epoch 4/10 | Loss: 2.1686
Epoch 5/10 | Loss: 1.9371
Epoch 6/10 | Loss: 1.6504
Epoch 7/10 | Loss: 1.2874
Epoch 8/10 | Loss: 1.0072
Epoch 9/10 | Loss: 0.8389
Epoch 10/10 | Loss: 0.5728


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cpu! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

In [6]:

# 测试函数
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            outputs = model(inputs.to("cuda:1"))
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets.to("cuda:1")).sum().item()
    
    return correct / total

# 测试准确率
test_acc = evaluate(model, loader)
print(f"Test Accuracy: {test_acc:.2%}")

Test Accuracy: 93.00%


In [3]:
def p_count(m):
    ttp=0
    tp=0
    for p in m.parameters():
        c=p.numel()
        if p.requires_grad == True:
            ttp+=c
        tp+=c
    print(f"Total trainable parameters: {ttp}")
    print(f"Total parameters: {tp}")

p_count(model)

Total trainable parameters: 309386
Total parameters: 309386


In [2]:

from datasets import Dataset, load_dataset

# 使用DataCollatorForLanguageModeling处理因果语言建模
from transformers import (AutoConfig, AutoTokenizer,
                          DataCollatorForLanguageModeling, DefaultDataCollator,
                          Trainer, TrainingArguments)
import wandb

def p_count(m):
    ttp = 0
    tp = 0
    for p in m.parameters():
        c = p.numel()
        if p.requires_grad == True:
            ttp += c
        tp += c
    print(f"Total trainable parameters: {ttp}")
    print(f"Total parameters: {tp}")

# 初始化模型和数据集
model_path="./HyenaBase"
tokenizer = AutoTokenizer.from_pretrained(
    model_path, trust_remote_code=True
)


def pack(
    _tokenizer,
    max_length,
    padding="max_length",
    pad_to_multiple_of=None,
    return_tensors="pt",
):
    def padseq(line):
        inputs = tokenizer(
            line["sequence"], max_length=max_length, truncation=True, padding=padding
        )
        return inputs
    return padseq

func = pack(tokenizer, 2048, padding="max_length")
def initial_dataset(data_files,settype,output_path = "./data",perfix = "nvbi_virus_0.1"):
    dataset_temp = load_dataset("csv", data_files=data_files)
    dataset_temp = dataset_temp.map(func, batched=True, num_proc=128)["train"].remove_columns(["ID","sequence","Length","genome"])
    dataset_temp.save_to_disk(f"{output_path}/{perfix}/{settype}", num_proc=128)
    return dataset_temp

trainset=initial_dataset("/home/zhengyulong/models/HyenaModel/data/trainset_0.1.csv","trainset")
evalset=initial_dataset("/home/zhengyulong/models/HyenaModel/data/evalset_0.1.csv","evalset")



Saving the dataset (0/128 shards):   0%|          | 0/593074 [00:00<?, ? examples/s]

Saving the dataset (0/128 shards):   0%|          | 0/73965 [00:00<?, ? examples/s]

In [3]:
datacollator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # 使用因果语言建模
    pad_to_multiple_of=tokenizer.pad_token_id,  # 可选填充对齐
)
config = AutoConfig.from_pretrained(
    model_path,
    trust_remote_code=True,
    num_labels=12,
    classfier_depth=2
)
model =  LyraDNAForCausalLM(config)
p_count(model)

Total trainable parameters: 1535244
Total parameters: 1535244


In [20]:
model

LyraDNAForCausalLM(
  (lyra): LyraDNAModel(
    (embeddings): HyenaEmbeddings(
      (word_embeddings): Embedding(16, 256)
    )
    (backbone): Lyra(
      (encoder): Linear(in_features=256, out_features=256, bias=True)
      (pgc_layers): ModuleList(
        (0-7): 8 x PGC(
          (conv): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(1,), groups=256)
          (in_proj): Linear(in_features=256, out_features=256, bias=True)
          (norm): RMSNorm()
          (in_norm): RMSNorm()
          (out_proj): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (s4_layers): ModuleList(
        (0-3): 4 x S4D(
          (kernel): S4DKernel()
          (activation): GELU(approximate='none')
          (dropout): DropoutNd()
          (output_linear): Sequential(
            (0): Conv1d(256, 512, kernel_size=(1,), stride=(1,))
            (1): GLU(dim=-2)
          )
        )
      )
      (norms): ModuleL

In [4]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '1,2,3,4,5,6,7'

training_args = TrainingArguments(
    output_dir="/pf9550-bdp-A800/zhengyulong/lyradna/NCBIVirus0.1_Pretrain",
    evaluation_strategy="steps",
    gradient_checkpointing=False,
    eval_steps=20,
    save_steps=10,
    save_total_limit=10,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.1,
    num_train_epochs=10,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    neftune_noise_alpha=5.0,
    max_grad_norm=5,
    bf16=False,
    logging_steps=1,
    report_to="wandb",
    optim="adamw_apex_fused",
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=trainset,
    eval_dataset=evalset,
    data_collator=datacollator,
    # compute_metrics=compute_metrics,
)
trainer.train()


/home/zhengyulong/miniconda3/lib/python3.9/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-04-07 11:56:57,212] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/zhengyulong/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/zhengyulong/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-12.0/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/zhengyulong/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-12.0/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/zhengyulong/miniconda3/compiler_compat/ld: /usr/local/cuda-12.0/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/zhengyulong/miniconda3/compiler_compat/ld: /usr/local/cuda-12.0/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/zhengyulong/miniconda3/compiler_compat/ld: /usr/local/cuda-12.0/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/zhengyulong/miniconda3/compiler_compat/ld: /usr/local/cuda-12.0/

/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
20,2.775200,2.754015
40,2.722600,2.682250
60,2.612300,2.562389
80,2.459000,2.385509
100,2.287400,2.149915


/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather a

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f846c607070>> (for post_run_cell):


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
from models import LyraDNAForCausalLM,LyraDNAForSequenceClassification
model = LyraDNAForSequenceClassification.from_pretrained("/pf9550-bdp-A800/zhengyulong/lyradna/NCBIVirus0.1_Pretrain/checkpoint-5000")

ImportError: cannot import name 'LyraDNAForSequenceClassification' from 'models' (/home/zhengyulong/models/lyra/models.py)

In [2]:
model

LyraDNAForCausalLM(
  (lyra): LyraDNAModel(
    (embeddings): HyenaEmbeddings(
      (word_embeddings): Embedding(16, 256)
    )
    (backbone): Lyra(
      (encoder): Linear(in_features=256, out_features=256, bias=True)
      (pgc_layers): ModuleList(
        (0-7): 8 x PGC(
          (conv): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,), groups=128)
          (in_proj): Linear(in_features=256, out_features=256, bias=True)
          (norm): RMSNorm()
          (in_norm): RMSNorm()
          (out_proj): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (s4_layers): ModuleList(
        (0-3): 4 x S4D(
          (kernel): S4DKernel()
          (activation): GELU(approximate='none')
          (dropout): DropoutNd()
          (output_linear): Sequential(
            (0): Conv1d(256, 512, kernel_size=(1,), stride=(1,))
            (1): GLU(dim=-2)
          )
        )
      )
      (norms): ModuleL